# Instalación de Librerias


In [ ]:
!pip install gradio groq pandas tabulate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.3 MB/s eta 0:00:00


## Configuración Api Keys

In [ ]:
from google.colab import userdata
from groq import Groq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
client = Groq(api_key=GROQ_API_KEY)

# Prueba rápida
resp = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Hola, responde solo: estoy funcionando"}]
)
print("✅ Groq responde:", resp.choices[0].message.content)

✅ Groq responde: Ok


##  Carga de datos y motor analítico

In [ ]:
import pandas as pd

df_recordings = pd.read_csv("1_Data_Recordings_CLEAN.csv")
df_metrics    = pd.read_csv("Data_Metrics_clean.csv")

print("✅ Recordings:", df_recordings.shape)
print("✅ Metrics:   ", df_metrics.shape)

def get_top_pages(df=df_recordings, top_n=5):
    try:
        col = "direccion_url_entrada"
        resultado = df[col].value_counts().head(top_n).reset_index()
        resultado.columns = ["pagina", "visitas"]
        return resultado
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})

def get_abandono(df=df_recordings, top_n=5):
    try:
        col = "direccion_url_salida"
        resultado = df[col].value_counts().head(top_n).reset_index()
        resultado.columns = ["pagina_de_salida", "sesiones_terminadas"]
        return resultado
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})

def get_duracion_promedio(df=df_recordings):
    try:
        col = "duracion_sesion_segundos"
        return {
            "promedio_seg": round(df[col].mean(), 1),
            "maximo_seg":   round(df[col].max(), 1),
            "minimo_seg":   round(df[col].min(), 1)
        }
    except Exception as e:
        return {"error": str(e)}

def get_dispositivos(df=df_recordings):
    try:
        col = "dispositivo"
        resultado = df[col].value_counts(normalize=True).mul(100).round(1).reset_index()
        resultado.columns = ["dispositivo", "porcentaje"]
        return resultado
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})

def get_frustracion(df=df_recordings):
    try:
        col = "posible_frustracion"
        total      = len(df)
        frustradas = int((df[col] == 1).sum())
        return {
            "sesiones_total":  total,
            "con_frustracion": frustradas,
            "porcentaje":      round(frustradas / total * 100, 1)
        }
    except Exception as e:
        return {"error": str(e)}

def get_engagement(df=df_recordings):
    try:
        col = "standarized_engagement_score"
        return {
            "engagement_promedio": round(df[col].mean(), 4),
            "engagement_max":      round(df[col].max(), 4)
        }
    except Exception as e:
        return {"error": str(e)}

def get_abandono_rapido(df=df_recordings):
    try:
        col = "abandono_rapido"
        total     = len(df)
        abandonos = int((df[col] == 1).sum())
        return {
            "sesiones_total":  total,
            "abandono_rapido": abandonos,
            "porcentaje":      round(abandonos / total * 100, 1)
        }
    except Exception as e:
        return {"error": str(e)}

def get_trafico_externo(df=df_recordings):
    try:
        col     = "trafico_externo"
        total   = len(df)
        externo = int((df[col] == 1).sum())
        directo = total - externo
        return {
            "trafico_externo": externo,
            "trafico_directo": directo,
            "pct_externo":     round(externo / total * 100, 1),
            "pct_directo":     round(directo / total * 100, 1)
        }
    except Exception as e:
        return {"error": str(e)}

def get_trafico_por_hora(df=df_recordings):
    try:
        df["hora_int"] = pd.to_datetime(df["hora"], format="%H:%M:%S").dt.hour
        resultado = df.groupby("hora_int").size().reset_index()
        resultado.columns = ["hora", "sesiones"]
        resultado = resultado.sort_values("sesiones", ascending=False).head(5)
        return resultado
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})

def get_top_paises(df=df_recordings, top_n=10):
    try:
        col = "pais"
        resultado = df[col].value_counts().head(top_n).reset_index()
        resultado.columns = ["pais", "sesiones"]
        return resultado
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})

def get_segundos_por_pagina(df=df_recordings, top_n=8):
    try:
        resultado = df.groupby("direccion_url_entrada")["tiempo_por_pagina"].mean().round(1)
        resultado = resultado.sort_values(ascending=False).head(top_n).reset_index()
        resultado.columns = ["pagina", "segundos_promedio"]
        return resultado
    except Exception as e:
        return pd.DataFrame({"error": [str(e)]})

ANALYTICS_MAP = {
    "top_pages":           get_top_pages,
    "abandono":            get_abandono,
    "duracion":            get_duracion_promedio,
    "dispositivos":        get_dispositivos,
    "frustracion":         get_frustracion,
    "engagement":          get_engagement,
    "abandono_rapido":     get_abandono_rapido,
    "trafico_externo":     get_trafico_externo,
    "trafico_hora":        get_trafico_por_hora,
    "segundos_por_pagina": get_segundos_por_pagina,
    "paises":              get_top_paises,
}

print("\n🧪 Probando funciones...\n")
for nombre, funcion in ANALYTICS_MAP.items():
    try:
        funcion()
        print(f"  ✅ {nombre}: OK")
    except Exception as e:
        print(f"  ❌ {nombre}: {str(e)}")

print("\n✅ Motor analítico listo")

✅ Recordings: (65677, 31)
✅ Metrics:    (20595, 16)

🧪 Probando funciones...

  ✅ top_pages: OK
  ✅ abandono: OK
  ✅ duracion: OK
  ✅ dispositivos: OK
  ✅ frustracion: OK
  ✅ engagement: OK
  ✅ abandono_rapido: OK
  ✅ trafico_externo: OK
  ✅ trafico_hora: OK
  ✅ segundos_por_pagina: OK
  ✅ paises: OK

✅ Motor analítico listo


## Configuración de Gemini y prompts

In [ ]:
PROMPT_PRINCIPAL = """
Eres Isis, una experta en Marketing Digital y analítica web con más de 10 años de experiencia.
Trabajas para CloudLabs Learning y tu misión es ayudar al equipo de Marketing a entender
el comportamiento de sus usuarios web y tomar decisiones estratégicas rápidas.

Tienes acceso a los siguientes datos reales extraídos de Microsoft Clarity:
{datos}

Pregunta del usuario: {pregunta}

INSTRUCCIONES DE RESPUESTA:
Responde SIEMPRE en este formato exacto cuando la pregunta sea sobre datos:

📊 DATOS
[Presenta aquí los números y métricas relevantes de forma clara y ordenada]

🧠 INTERPRETACIÓN
[Explica qué significan esos datos para el negocio en máximo 3 oraciones simples,
sin tecnicismos, como si le hablaras a alguien que no conoce de tecnología]

💡 RECOMENDACIÓN
[Da una acción concreta y específica que el equipo de Marketing puede ejecutar
basándose en estos datos]

REGLAS IMPORTANTES:
- Responde siempre en español
- Usa emojis para hacer la respuesta más amigable y fácil de leer
- Si te preguntan quién eres, preséntate como Isis, experta en Marketing Digital de CloudLabs
- Si la pregunta no tiene relación con marketing o datos web, redirígela amablemente
- Nunca inventes datos que no estén en el contexto proporcionado
- Sé directa y accionable, el equipo necesita respuestas que les ahorren tiempo
"""

print("✅ Prompt mejorado listo")


✅ Prompt mejorado listo


## Funciones del co-piloto


In [ ]:
def llamar_llm(prompt: str) -> str:
    resp = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500
    )
    return resp.choices[0].message.content.strip()

def calcular_todos_los_datos() -> str:
    resumen = []

    try:
        top = get_top_pages()
        resumen.append("PAGINAS MAS VISITADAS:\n" + top.to_string(index=False))
    except Exception as e:
        resumen.append(f"PAGINAS MAS VISITADAS: No disponible ({e})")

    try:
        abandono = get_abandono()
        resumen.append("PAGINAS CON MAYOR ABANDONO:\n" + abandono.to_string(index=False))
    except Exception as e:
        resumen.append(f"PAGINAS CON MAYOR ABANDONO: No disponible ({e})")

    try:
        duracion = get_duracion_promedio()
        resumen.append("DURACION DE SESION:\n" + "\n".join([f"{k}: {v}" for k, v in duracion.items()]))
    except Exception as e:
        resumen.append(f"DURACION DE SESION: No disponible ({e})")

    try:
        dispositivos = get_dispositivos()
        resumen.append("DISPOSITIVOS:\n" + dispositivos.to_string(index=False))
    except Exception as e:
        resumen.append(f"DISPOSITIVOS: No disponible ({e})")

    try:
        frustracion = get_frustracion()
        resumen.append("FRUSTRACION:\n" + "\n".join([f"{k}: {v}" for k, v in frustracion.items()]))
    except Exception as e:
        resumen.append(f"FRUSTRACION: No disponible ({e})")

    try:
        engagement = get_engagement()
        resumen.append("ENGAGEMENT:\n" + "\n".join([f"{k}: {v}" for k, v in engagement.items()]))
    except Exception as e:
        resumen.append(f"ENGAGEMENT: No disponible ({e})")

    return "\n\n".join(resumen)

DATOS_GLOBALES = calcular_todos_los_datos()
print("✅ Datos precalculados:")
print(DATOS_GLOBALES[:300])

def responder(mensaje: str, historial) -> str:
    if not mensaje.strip():
        return "Por favor escribe una pregunta."
    try:
        # Construir mensajes con historial completo
        messages = [
            {
                "role": "system",
                "content": PROMPT_PRINCIPAL.format(
                    datos=DATOS_GLOBALES,
                    pregunta=""
                )
            }
        ]

        # Agregar historial previo
        for humano, asistente in historial:
            messages.append({"role": "user",      "content": humano})
            messages.append({"role": "assistant", "content": asistente})

        # Agregar mensaje actual
        messages.append({"role": "user", "content": mensaje})

        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=messages,
            max_tokens=500
        )
        return resp.choices[0].message.content.strip()

    except Exception as e:
        return f"Error: {str(e)}"

print("✅ Función con memoria lista")

print("\n✅ Funciones listas")

✅ Datos precalculados:
PAGINAS MAS VISITADAS: No disponible (name 'get_top_pages' is not defined)

PAGINAS CON MAYOR ABANDONO: No disponible (name 'get_abandono' is not defined)

DURACION DE SESION: No disponible (name 'get_duracion_promedio' is not defined)

DISPOSITIVOS: No disponible (name 'get_dispositivos' is not def
✅ Función con memoria lista

✅ Funciones listas


##  Interfaz Gradio

In [ ]:
!pip install plotly -q

In [ ]:
import gradio as gr
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# ─── Semáforo ─────────────────────────────────────────────────────────────────
def semaforo_html(url_seleccionada="Todas"):
    if url_seleccionada == "Todas":
        df_fil = df_recordings.copy()
    else:
        df_fil = df_recordings[df_recordings["direccion_url_entrada"] == url_seleccionada].copy()

    frus = get_frustracion(df=df_fil)
    eng  = get_engagement(df=df_fil)
    aban = get_abandono_rapido(df=df_fil)
    traf = get_trafico_externo(df=df_fil)
    dur  = get_duracion_promedio(df=df_fil)

    def estado(valor, umbrales, invertido=False):
        verde, amarillo = umbrales
        if not invertido:
            if valor >= verde:      return "#3CC13B", "🟢", "Bueno"
            elif valor >= amarillo: return "#EF9F27", "🟡", "Moderado"
            else:                   return "#D83A2A", "🔴", "Bajo"
        else:
            if valor <= verde:      return "#3CC13B", "🟢", "Bueno"
            elif valor <= amarillo: return "#EF9F27", "🟡", "Moderado"
            else:                   return "#D83A2A", "🔴", "Alto"

    color_eng,  icon_eng,  label_eng  = estado(eng["engagement_promedio"],  [5, 1],    invertido=False)
    color_frus, icon_frus, label_frus = estado(frus["porcentaje"],           [20, 40],  invertido=True)
    color_aban, icon_aban, label_aban = estado(aban["porcentaje"],           [30, 50],  invertido=True)
    color_traf, icon_traf, label_traf = estado(traf["pct_externo"],          [40, 20],  invertido=False)
    color_dur,  icon_dur,  label_dur  = estado(dur["promedio_seg"],          [120, 30], invertido=False)

    subtitulo = "todas las páginas" if url_seleccionada == "Todas" else url_seleccionada.replace("https://cloudlabslearning.com", "")

    def tarjeta(icon, label, titulo, valor, color, descripcion):
        return f"""
        <div style="background:white; border-radius:12px; padding:16px;
                    border-left:4px solid {color};
                    box-shadow:0 2px 8px rgba(27,58,107,0.08);
                    display:flex; align-items:center; gap:14px;">
            <div style="font-size:28px;">{icon}</div>
            <div style="flex:1;">
                <div style="color:#4A6FA5; font-size:11px; font-weight:600;
                            letter-spacing:1px; margin-bottom:2px;">{titulo.upper()}</div>
                <div style="color:#1B3A6B; font-size:15px; font-weight:700;">{valor}</div>
                <div style="color:#4A6FA5; font-size:11px;">{descripcion}</div>
            </div>
            <div style="background:{color}22; border-radius:20px; padding:4px 10px;
                        color:{color}; font-size:12px; font-weight:700;">{label}</div>
        </div>"""

    return f"""
    <div style="margin-bottom:16px;">
        <div style="color:#1B3A6B; font-size:14px; font-weight:700;
                    margin-bottom:10px; display:flex; align-items:center; gap:8px;">
            🏥 Salud del sitio web
            <span style="font-size:11px; color:#4A6FA5; font-weight:400;">— {subtitulo}</span>
        </div>
        <div style="display:grid; grid-template-columns:repeat(5,1fr); gap:12px;">
            {tarjeta(icon_eng,  label_eng,  "Engagement",      f"{eng['engagement_promedio']}%",  color_eng,  "score promedio")}
            {tarjeta(icon_frus, label_frus, "Frustración",     f"{frus['porcentaje']}%",           color_frus, "de sesiones")}
            {tarjeta(icon_aban, label_aban, "Abandono rápido", f"{aban['porcentaje']}%",           color_aban, "de sesiones")}
            {tarjeta(icon_traf, label_traf, "Tráfico externo", f"{traf['pct_externo']}%",          color_traf, "viene de fuera")}
            {tarjeta(icon_dur,  label_dur,  "Seg. por página", f"{dur['promedio_seg']}s",          color_dur,  "tiempo promedio")}
        </div>
    </div>"""

# ─── Feedback de Isis ─────────────────────────────────────────────────────────
def isis_feedback(url_seleccionada="Todas"):
    if url_seleccionada == "Todas":
        df_fil = df_recordings.copy()
    else:
        df_fil = df_recordings[df_recordings["direccion_url_entrada"] == url_seleccionada].copy()

    frus = get_frustracion(df=df_fil)
    eng  = get_engagement(df=df_fil)
    aban = get_abandono_rapido(df=df_fil)
    traf = get_trafico_externo(df=df_fil)
    dur  = get_duracion_promedio(df=df_fil)

    filtro = "todas las páginas" if url_seleccionada == "Todas" else url_seleccionada.replace("https://cloudlabslearning.com", "")

    prompt = f"""
Eres Isis, experta en Marketing Digital de CloudLabs Learning.
Analiza estos indicadores del sitio web para: {filtro}

INDICADORES:
- Engagement: {eng['engagement_promedio']}%
- Tasa de frustración: {frus['porcentaje']}% de sesiones
- Abandono rápido: {aban['porcentaje']}% de sesiones
- Tráfico externo: {traf['pct_externo']}%
- Duración promedio de sesión: {dur['promedio_seg']} segundos

Da un diagnóstico ejecutivo con:
1. Estado general del sitio en una sola frase
2. El indicador más preocupante y por qué
3. Dos acciones concretas que el equipo de Marketing puede tomar esta semana

Responde en español, de forma clara, directa y con emojis. Máximo 150 palabras.
"""
    try:
        resp = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300
        )
        texto = resp.choices[0].message.content.strip()
        return f"""
        <div style="background:white; border-radius:12px; padding:20px;
                    border-left:4px solid #2D6BE4;
                    box-shadow:0 2px 8px rgba(27,58,107,0.08); margin-top:12px;">
            <div style="color:#1B3A6B; font-size:13px; font-weight:700;
                        margin-bottom:10px; display:flex; align-items:center; gap:8px;">
                🤖 Diagnóstico de Isis
                <span style="font-size:11px; color:#4A6FA5; font-weight:400;">— {filtro}</span>
            </div>
            <div style="color:#1B3A6B; font-size:13px; line-height:1.7;">
                {texto.replace(chr(10), '<br>')}
            </div>
        </div>"""
    except Exception as e:
        return f"<div style='color:red;'>Error: {str(e)}</div>"

# ─── Gráficas base ────────────────────────────────────────────────────────────
def grafica_top_pages():
    df = get_top_pages(top_n=8)
    df["pagina_corta"] = df["pagina"].str.replace("https://cloudlabslearning.com", "", regex=False)
    df["pagina_corta"] = df["pagina_corta"].apply(lambda x: x[:40] + "..." if len(x) > 40 else x)
    df["pagina_corta"] = df["pagina_corta"].replace("", "/")
    fig = px.bar(df, x="visitas", y="pagina_corta", orientation="h",
                 color="visitas", color_continuous_scale=["#A8C8FF", "#1B3A6B"],
                 title="Páginas más visitadas", text="visitas")
    fig.update_traces(textposition="outside", textfont_size=11)
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                      font_family="Segoe UI", font_size=11,
                      title_font_size=14, title_font_color="#1B3A6B",
                      showlegend=False, coloraxis_showscale=False,
                      margin=dict(l=180, r=60, t=40, b=20), height=320,
                      yaxis=dict(autorange="reversed", tickfont=dict(size=11), title=""),
                      xaxis=dict(title="Visitas", tickfont=dict(size=10)))
    return fig

def grafica_abandono():
    df = get_abandono(top_n=8)
    df["pagina_corta"] = df["pagina_de_salida"].str.replace("https://cloudlabslearning.com", "", regex=False)
    df["pagina_corta"] = df["pagina_corta"].apply(lambda x: x[:40] + "..." if len(x) > 40 else x)
    df["pagina_corta"] = df["pagina_corta"].replace("", "/")
    fig = px.bar(df, x="sesiones_terminadas", y="pagina_corta", orientation="h",
                 color="sesiones_terminadas", color_continuous_scale=["#FFD0CC", "#D83A2A"],
                 title="Páginas con mayor abandono", text="sesiones_terminadas")
    fig.update_traces(textposition="outside", textfont_size=11)
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                      font_family="Segoe UI", font_size=11,
                      title_font_size=14, title_font_color="#1B3A6B",
                      showlegend=False, coloraxis_showscale=False,
                      margin=dict(l=180, r=60, t=40, b=20), height=320,
                      yaxis=dict(autorange="reversed", tickfont=dict(size=11), title=""),
                      xaxis=dict(title="Sesiones terminadas", tickfont=dict(size=10)))
    return fig



def grafica_dispositivos():
    df = get_dispositivos()
    fig = px.pie(df, values="porcentaje", names="dispositivo",
                 title="Distribución por dispositivo",
                 color_discrete_sequence=["#1B3A6B", "#2D6BE4", "#3CC13B", "#A8C8FF"])
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                      font_family="Segoe UI", title_font_size=14,
                      title_font_color="#1B3A6B",
                      margin=dict(l=10, r=10, t=40, b=10), height=320)
    return fig

def grafica_engagement():
    eng  = get_engagement()
    frus = get_frustracion()
    fig  = go.Figure()
    fig.add_trace(go.Indicator(
        mode="gauge+number", value=eng["engagement_promedio"],
        number={"suffix": "%", "valueformat": ".2f"},
        title={"text": "Engagement promedio", "font": {"size": 13}},
        gauge={"axis": {"range": [0, eng["engagement_max"]]}, "bar": {"color": "#2D6BE4"},
               "steps": [{"range": [0, eng["engagement_max"]*0.4], "color": "#EBF4FF"},
                         {"range": [eng["engagement_max"]*0.4, eng["engagement_max"]*0.7], "color": "#A8C8FF"},
                         {"range": [eng["engagement_max"]*0.7, eng["engagement_max"]], "color": "#D0E4FF"}]},
        domain={"x": [0, 0.5], "y": [0, 1]}
    ))
    fig.add_trace(go.Indicator(
        mode="gauge+number", value=frus["porcentaje"],
        number={"suffix": "%"},
        title={"text": "Tasa de frustración", "font": {"size": 13}},
        gauge={"axis": {"range": [0, 100]}, "bar": {"color": "#D83A2A"},
               "steps": [{"range": [0, 30], "color": "#EBF4FF"},
                         {"range": [30, 60], "color": "#FFD0CC"},
                         {"range": [60, 100], "color": "#FFADA6"}]},
        domain={"x": [0.5, 1], "y": [0, 1]}
    ))
    fig.update_layout(paper_bgcolor="white", font_family="Segoe UI",
                      height=280, margin=dict(l=20, r=20, t=40, b=10))
    return fig

def grafica_abandono_por_url(url_seleccionada="Todas"):
    df_rec = df_recordings.copy()
    if url_seleccionada is None or url_seleccionada == "Todas":
        df_filtrado = df_rec
        titulo = "Abandono general — todas las páginas"
    else:
        df_filtrado = df_rec[df_rec["direccion_url_entrada"] == url_seleccionada]
        titulo = f"Abandono desde: {url_seleccionada.replace('https://cloudlabslearning.com', '')}"

    resultado = df_filtrado["direccion_url_salida"].value_counts().head(8).reset_index()
    resultado.columns = ["pagina_salida", "sesiones"]
    resultado["pagina_corta"] = resultado["pagina_salida"].str.replace(
        "https://cloudlabslearning.com", "", regex=False)
    resultado["pagina_corta"] = resultado["pagina_corta"].apply(
        lambda x: x[:40] + "..." if len(x) > 40 else x)
    resultado["pagina_corta"] = resultado["pagina_corta"].replace("", "/")

    fig = px.bar(resultado, x="sesiones", y="pagina_corta", orientation="h",
                 color="sesiones", color_continuous_scale=["#FFD0CC", "#D83A2A"],
                 title=titulo, text="sesiones")
    fig.update_traces(textposition="outside", textfont_size=11)
    fig.update_layout(plot_bgcolor="white", paper_bgcolor="white",
                      font_family="Segoe UI", font_size=11,
                      title_font_size=13, title_font_color="#1B3A6B",
                      showlegend=False, coloraxis_showscale=False,
                      margin=dict(l=180, r=60, t=40, b=20), height=340,
                      yaxis=dict(autorange="reversed", tickfont=dict(size=11), title=""),
                      xaxis=dict(title="Sesiones terminadas", tickfont=dict(size=10)))
    return fig

# ─── Gráficas diferenciadoras ─────────────────────────────────────────────────

# 1. Mapa por país
def grafica_mapa_paises():
    try:
        df = df_recordings.copy()
        paises = df["pais"].value_counts().reset_index()
        paises.columns = ["pais", "sesiones"]
        fig = px.choropleth(
            paises,
            locations="pais",
            locationmode="country names",
            color="sesiones",
            color_continuous_scale=["#EBF4FF", "#A8C8FF", "#2D6BE4", "#1B3A6B"],
            title="🌍 Tráfico por país"
        )
        fig.update_layout(
            paper_bgcolor="white",
            font_family="Segoe UI",
            title_font_size=14,
            title_font_color="#1B3A6B",
            margin=dict(l=0, r=0, t=40, b=0),
            height=380,
            geo=dict(showframe=False, showcoastlines=True,
                     projection_type="natural earth",
                     bgcolor="white")
        )
        return fig
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(text=f"Error: {str(e)}", showarrow=False)
        return fig

# 2. Frustración por dispositivo
def grafica_frustracion_dispositivo():
    try:
        df = df_recordings.copy()
        resumen = df.groupby("dispositivo").agg(
            total=("posible_frustracion", "count"),
            frustrados=("posible_frustracion", "sum")
        ).reset_index()
        resumen["pct_frustracion"] = (resumen["frustrados"] / resumen["total"] * 100).round(1)

        fig = px.bar(
            resumen, x="dispositivo", y="pct_frustracion",
            color="pct_frustracion",
            color_continuous_scale=["#EBF4FF", "#FFD0CC", "#D83A2A"],
            title="😤 Frustración por dispositivo (%)",
            text="pct_frustracion"
        )
        fig.update_traces(texttemplate="%{text}%", textposition="outside", textfont_size=12)
        fig.update_layout(
            plot_bgcolor="white", paper_bgcolor="white",
            font_family="Segoe UI", font_size=11,
            title_font_size=14, title_font_color="#1B3A6B",
            showlegend=False, coloraxis_showscale=False,
            margin=dict(l=20, r=20, t=40, b=20), height=320,
            xaxis=dict(title="Dispositivo", tickfont=dict(size=11)),
            yaxis=dict(title="% con frustración", tickfont=dict(size=10))
        )
        return fig
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(text=f"Error: {str(e)}", showarrow=False)
        return fig

# 3. Tendencia de sesiones en el tiempo
def grafica_tendencia_tiempo():
    try:
        df = df_recordings.copy()
        df["fecha"] = pd.to_datetime(df["fecha"])
        tendencia = df.groupby("fecha").size().reset_index()
        tendencia.columns = ["fecha", "sesiones"]
        tendencia = tendencia.sort_values("fecha")

        fig = px.line(
            tendencia, x="fecha", y="sesiones",
            title="📈 Tendencia de sesiones en el tiempo",
            markers=True
        )
        fig.update_traces(
            line_color="#2D6BE4",
            line_width=2,
            marker_color="#1B3A6B",
            marker_size=4
        )
        fig.add_trace(go.Scatter(
            x=tendencia["fecha"],
            y=[tendencia["sesiones"].mean()] * len(tendencia),
            mode="lines",
            line=dict(color="#EF9F27", dash="dash", width=1.5),
            name="Promedio"
        ))
        fig.update_layout(
            plot_bgcolor="white", paper_bgcolor="white",
            font_family="Segoe UI", font_size=11,
            title_font_size=14, title_font_color="#1B3A6B",
            margin=dict(l=20, r=20, t=40, b=20), height=320,
            xaxis=dict(title="Fecha", tickfont=dict(size=10)),
            yaxis=dict(title="Sesiones", tickfont=dict(size=10)),
            legend=dict(orientation="h", yanchor="bottom", y=1.02)
        )
        return fig
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(text=f"Error: {str(e)}", showarrow=False)
        return fig

# 4. Flujo Sankey
def grafica_sankey():
    try:
        df = df_recordings.copy()
        df["entrada_corta"] = df["direccion_url_entrada"].str.replace(
            "https://cloudlabslearning.com", "", regex=False)
        df["entrada_corta"] = df["entrada_corta"].apply(
            lambda x: x[:30] + "..." if len(x) > 30 else x).replace("", "/")
        df["salida_corta"] = df["direccion_url_salida"].str.replace(
            "https://cloudlabslearning.com", "", regex=False)
        df["salida_corta"] = df["salida_corta"].apply(
            lambda x: x[:30] + "..." if len(x) > 30 else x).replace("", "/")

        flujo = df.groupby(["entrada_corta", "salida_corta"]).size().reset_index()
        flujo.columns = ["entrada", "salida", "sesiones"]
        flujo = flujo.sort_values("sesiones", ascending=False).head(20)

        entradas = flujo["entrada"].unique().tolist()
        salidas  = [s + " " for s in flujo["salida"].unique().tolist()]
        nodos    = entradas + salidas

        idx_entrada = {v: i for i, v in enumerate(entradas)}
        idx_salida  = {v: i + len(entradas) for i, v in enumerate(salidas)}

        fig = go.Figure(go.Sankey(
            node=dict(
                pad=15, thickness=20,
                line=dict(color="#1B3A6B", width=0.5),
                label=nodos,
                color=["#2D6BE4"] * len(entradas) + ["#A8C8FF"] * len(salidas)
            ),
            link=dict(
                source=[idx_entrada[r["entrada"]] for _, r in flujo.iterrows()],
                target=[idx_salida[r["salida"] + " "] for _, r in flujo.iterrows()],
                value=flujo["sesiones"].tolist(),
                color="rgba(45,107,228,0.2)"
            )
        ))
        return fig
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(text=f"Error: {str(e)}", showarrow=False)
        return fig

# 5. Segmentación de usuarios
def grafica_segmentacion():
    try:
        df = df_recordings.copy()

        def segmentar(row):
            if row["standarized_engagement_score"] > 0.1 and row["posible_frustracion"] == 0:
                return "👤 Usuario comprometido"
            elif row["posible_frustracion"] == 1:
                return "😤 Usuario frustrado"
            elif row["abandono_rapido"] == 1:
                return "💨 Visitante de paso"
            else:
                return "🔍 Explorador casual"

        df["segmento"] = df.apply(segmentar, axis=1)
        resumen = df["segmento"].value_counts().reset_index()
        resumen.columns = ["segmento", "cantidad"]
        resumen["porcentaje"] = (resumen["cantidad"] / resumen["cantidad"].sum() * 100).round(1)

        fig = px.bar(
            resumen, x="segmento", y="porcentaje",
            color="segmento",
            color_discrete_sequence=["#3CC13B", "#D83A2A", "#EF9F27", "#2D6BE4"],
            title="🎯 Segmentación de usuarios por comportamiento",
            text="porcentaje"
        )
        fig.update_traces(texttemplate="%{text}%", textposition="outside", textfont_size=12)
        fig.update_layout(
            plot_bgcolor="white", paper_bgcolor="white",
            font_family="Segoe UI", font_size=11,
            title_font_size=14, title_font_color="#1B3A6B",
            showlegend=False,
            margin=dict(l=20, r=20, t=40, b=20), height=340,
            xaxis=dict(title="", tickfont=dict(size=11)),
            yaxis=dict(title="% de usuarios", tickfont=dict(size=10))
        )
        return fig
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(text=f"Error: {str(e)}", showarrow=False)
        return fig

# 6. Heatmap tráfico por día y hora
def grafica_heatmap_trafico():
    try:
        df = df_recordings.copy()
        df["hora_int"] = pd.to_datetime(df["hora"], format="%H:%M:%S").dt.hour
        df["dia_semana"] = pd.to_datetime(df["fecha"]).dt.day_name()

        orden_dias = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
        nombres_dias = {"Monday":"Lun","Tuesday":"Mar","Wednesday":"Mié",
                        "Thursday":"Jue","Friday":"Vie","Saturday":"Sáb","Sunday":"Dom"}

        pivot = df.groupby(["hora_int","dia_semana"]).size().reset_index()
        pivot.columns = ["hora","dia","sesiones"]
        pivot["dia"] = pd.Categorical(pivot["dia"], categories=orden_dias, ordered=True)
        pivot["dia_nombre"] = pivot["dia"].map(nombres_dias)
        pivot = pivot.sort_values("dia")

        fig = px.density_heatmap(
            pivot, x="dia_nombre", y="hora", z="sesiones",
            color_continuous_scale=["#EBF4FF","#A8C8FF","#2D6BE4","#1B3A6B"],
            title="🗓️ Mapa de calor — Tráfico por día y hora",
            labels={"dia_nombre":"Día","hora":"Hora","sesiones":"Sesiones"}
        )
        fig.update_layout(
            plot_bgcolor="white", paper_bgcolor="white",
            font_family="Segoe UI", font_size=11,
            title_font_size=14, title_font_color="#1B3A6B",
            margin=dict(l=60, r=20, t=40, b=40), height=380,
            xaxis=dict(categoryorder="array",
                       categoryarray=["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"],
                       title="Día de la semana", tickfont=dict(size=11)),
            yaxis=dict(title="Hora del día", tickfont=dict(size=10),
                       tickmode="linear", tick0=0, dtick=2),
            coloraxis_colorbar=dict(title="Sesiones")
        )
        return fig
    except Exception as e:
        fig = go.Figure()
        fig.add_annotation(text=f"Error: {str(e)}", showarrow=False)
        return fig

def get_urls_disponibles():
    return ["Todas"] + df_recordings["direccion_url_entrada"].value_counts().head(20).index.tolist()

def kpis_html():
    dur  = get_duracion_promedio()
    frus = get_frustracion()
    eng  = get_engagement()
    top  = get_top_pages(top_n=1)
    pagina_top = top["pagina"].iloc[0].replace("https://cloudlabslearning.com", "")
    pagina_top = pagina_top if pagina_top else "/"
    if len(pagina_top) > 20:
        pagina_top = pagina_top[:20] + "..."
    return f"""
    <div style="display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin-bottom:16px;">
        <div style="background:white; border-radius:12px; padding:16px;
                    border-left:4px solid #2D6BE4; box-shadow:0 2px 8px rgba(27,58,107,0.08);">
            <div style="color:#4A6FA5; font-size:11px; font-weight:600; letter-spacing:1px; margin-bottom:6px;">DURACIÓN PROMEDIO</div>
            <div style="color:#1B3A6B; font-size:24px; font-weight:700;">{int(dur['promedio_seg'])}s</div>
            <div style="color:#4A6FA5; font-size:11px;">por sesión</div>
        </div>
        <div style="background:white; border-radius:12px; padding:16px;
                    border-left:4px solid #3CC13B; box-shadow:0 2px 8px rgba(27,58,107,0.08);">
            <div style="color:#4A6FA5; font-size:11px; font-weight:600; letter-spacing:1px; margin-bottom:6px;">ENGAGEMENT</div>
            <div style="color:#1B3A6B; font-size:24px; font-weight:700;">{eng['engagement_promedio']}%</div>
            <div style="color:#4A6FA5; font-size:11px;">score promedio</div>
        </div>
        <div style="background:white; border-radius:12px; padding:16px;
                    border-left:4px solid #D83A2A; box-shadow:0 2px 8px rgba(27,58,107,0.08);">
            <div style="color:#4A6FA5; font-size:11px; font-weight:600; letter-spacing:1px; margin-bottom:6px;">FRUSTRACIÓN</div>
            <div style="color:#1B3A6B; font-size:24px; font-weight:700;">{frus['porcentaje']}%</div>
            <div style="color:#4A6FA5; font-size:11px;">de sesiones</div>
        </div>
        <div style="background:white; border-radius:12px; padding:16px;
                    border-left:4px solid #EF9F27; box-shadow:0 2px 8px rgba(27,58,107,0.08);">
            <div style="color:#4A6FA5; font-size:11px; font-weight:600; letter-spacing:1px; margin-bottom:6px;">PÁGINA TOP</div>
            <div style="color:#1B3A6B; font-size:16px; font-weight:700;">{pagina_top}</div>
            <div style="color:#4A6FA5; font-size:11px;">más visitada</div>
        </div>
    </div>"""

# ─── CSS ──────────────────────────────────────────────────────────────────────
css = """
body, .gradio-container { background-color: #F0F6FF !important; }
.gradio-container { max-width: 1100px !important; margin: auto !important; font-family: 'Segoe UI', sans-serif !important; }
footer { display: none !important; }
.tab-nav button { color: #4A6FA5 !important; font-weight: 600 !important; }
.tab-nav button.selected { color: #1B3A6B !important; border-bottom: 2px solid #3CC13B !important; }
"""

# ─── Interfaz ─────────────────────────────────────────────────────────────────
with gr.Blocks(css=css, title="Isis — Co-piloto de Marketing") as demo:

    gr.HTML("""
        <div style="background:linear-gradient(135deg,#1B3A6B 0%,#2D6BE4 100%);
                    border-radius:16px; padding:20px 24px;
                    display:flex; align-items:center; gap:16px; margin-bottom:16px;">
            <div style="background:white; border-radius:50%; width:56px; height:56px;
                        display:flex; align-items:center; justify-content:center;
                        font-size:28px; flex-shrink:0;">🤖</div>
            <div>
                <div style="color:white; font-size:22px; font-weight:700;">Isis</div>
                <div style="color:#A8C8FF; font-size:13px;">Co-piloto de Marketing · CloudLabs Learning</div>
            </div>
            <div style="margin-left:auto;">
                <span style="background:#3CC13B; border-radius:20px; padding:5px 12px;
                             color:white; font-size:12px; font-weight:600;">● En línea</span>
            </div>
        </div>
    """)

    with gr.Tabs():

        with gr.Tab("📊 Dashboard"):

            url_global = gr.Dropdown(
                choices=get_urls_disponibles(),
                value="Todas",
                label="🌐 Filtrar dashboard por URL de entrada",
            )

            semaforo = gr.HTML(value=semaforo_html("Todas"))
            feedback = gr.HTML(value="")

            btn_feedback = gr.Button("🤖 Pedir diagnóstico a Isis", variant="secondary")

            kpis = gr.HTML(value=kpis_html())

            with gr.Row():
                plot_top      = gr.Plot(value=grafica_top_pages(),   label="")
                plot_abandono = gr.Plot(value=grafica_abandono(),     label="")

            with gr.Row():
                plot_disp = gr.Plot(value=grafica_dispositivos(),        label="")
                plot_eng  = gr.Plot(value=grafica_engagement(),           label="")

            with gr.Row():
                plot_mapa     = gr.Plot(value=grafica_mapa_paises(),      label="")
                plot_tend     = gr.Plot(value=grafica_tendencia_tiempo(),  label="")

            with gr.Row():
                plot_frus_disp = gr.Plot(value=grafica_frustracion_dispositivo(), label="")

            with gr.Row():
                plot_heatmap = gr.Plot(value=grafica_heatmap_trafico(), label="")


            btn_feedback.click(
                fn=isis_feedback,
                inputs=[url_global],
                outputs=[feedback]
            )

            btn_refresh = gr.Button("🔄 Actualizar datos", variant="secondary")
            btn_refresh.click(
                fn=lambda: (
                    semaforo_html("Todas"), "",
                    kpis_html(),
                    grafica_top_pages(), grafica_abandono(),
                    grafica_dispositivos(), grafica_engagement(),
                    grafica_mapa_paises(), grafica_tendencia_tiempo(),
                    grafica_frustracion_dispositivo(), grafica_segmentacion(),
                    grafica_heatmap_trafico(), grafica_sankey(),
                    grafica_abandono_por_url("Todas")
                ),
                outputs=[semaforo, feedback, kpis,
                         plot_top, plot_abandono,
                         plot_disp, plot_eng,
                         plot_mapa, plot_tend,
                         plot_frus_disp,
                         plot_heatmap,
            ])

        with gr.Tab("🤖 Hablar con Isis"):

            chatbot = gr.Chatbot(
                label="", height=440,
                bubble_full_width=False,
                show_copy_button=True,
                avatar_images=(
                    None,
                    "https://api.dicebear.com/7.x/bottts/svg?seed=Isis&backgroundColor=1B3A6B"
                )
            )

            with gr.Row():
                txt_input = gr.Textbox(
                    placeholder="Pregúntale a Isis sobre tus datos...",
                    label="", scale=5, autofocus=True, container=False
                )
                btn_enviar = gr.Button("Enviar →", variant="primary", scale=1)

            gr.Examples(
                examples=[
                    "¿Cuáles son las páginas más visitadas?",
                    "¿En qué páginas abandonan más los usuarios?",
                    "¿Cuánto tiempo pasan en promedio los usuarios?",
                    "¿Desde qué dispositivos acceden más?",
                    "¿Qué porcentaje de usuarios muestra frustración?",
                    "¿Cómo está el nivel de engagement general?",
                ],
                inputs=txt_input,
                label="Preguntas sugeridas"
            )

            estado_historial = gr.State([])

            def responder_ui(mensaje, historial):
                respuesta = responder(mensaje, historial)
                historial.append((mensaje, respuesta))
                return "", historial

            btn_enviar.click(
                fn=responder_ui,
                inputs=[txt_input, estado_historial],
                outputs=[txt_input, chatbot]
            )
            txt_input.submit(
                fn=responder_ui,
                inputs=[txt_input, estado_historial],
                outputs=[txt_input, chatbot]
            )

    gr.HTML("""
        <div style="text-align:center; padding:10px 0; color:#4A6FA5; font-size:12px;">
            Isis · Powered by Groq · CloudLabs Learning © 2026
        </div>
    """)

demo.launch(share=True, debug=True)

/tmp/ipykernel_18028/168708544.py:521: DeprecationWarning:

The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.

/tmp/ipykernel_18028/168708544.py:605: UserWarning:

You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.

/tmp/ipykernel_18028/168708544.py:605: DeprecationWarning:

The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.

/tmp/ipykernel_18028/168708544.py:605: DeprecationWarning:

The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.

/tmp/ipykernel_18028/168708544.py:605: DeprecationWarning:

The default value of 'allow_tags' in gr.Chatbot will be cha

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://aed74177e1d01a3c0a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print(DATOS_GLOBALES)

In [ ]:
import gradio as gr
print(gr.__version__)